<a href="https://colab.research.google.com/github/Zohaibarif69/flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zohaibarif69/flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use Random Forest classification for this modeling lane.

The Week-4 baseline is a rule-based score using observed search visibility, CTR, and AI referral activity. A Random Forest is appropriate because it can capture non-linear relationships between these signals without requiring a linear relationship.

The model is used for decision-support rather than causal inference. I will compare it with the Week-4 baseline on the same held-out data and the same evaluation metric. If the model does not provide a meaningful measured improvement, the simpler baseline remains preferable.

In [ ]:
import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata
from huggingface_hub import hf_hub_download

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", HF_TOKEN is not None)

HF_TOKEN available: True


In [ ]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_path = march_file.replace("\\", "/")

march = duckdb.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_ai
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
""").df()

print("March rows:", len(march))
display(march.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_ai
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>


In [ ]:
available_months = []

for month in ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]:
    try:
        test_file = hf_hub_download(
            repo_id="FlyRank/internship-warehouse",
            filename=f"fact_content_daily_performance/month={month}/data_0.parquet",
            repo_type="dataset",
            token=HF_TOKEN
        )
        available_months.append(month)
    except Exception:
        pass

print("Available months:", available_months)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  149MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Available months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


## 2. Split design

I will use a time-aware split.

The model features come from an earlier observation period, while the outcome is measured in a later period. This is more honest for the decision-support question because information from the future should not be available when making the original prioritization decision.

The Week-4 baseline and the Random Forest will be evaluated on the same held-out observations and the same outcome definition.

In [ ]:
# Use the first available month after March as the future evaluation period.
future_months = [
    m for m in available_months
    if m > "2026-03"
]

assert len(future_months) > 0, (
    "No later warehouse month was found. "
    "Do not invent a future target."
)

future_month = future_months[0]

print("Feature period: 2026-03")
print("Future evaluation period:", future_month)

Feature period: 2026-03
Future evaluation period: 2026-04


In [ ]:
future_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=f"fact_content_daily_performance/month={future_month}/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

future_path = future_file.replace("\\", "/")

future = duckdb.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_ai
FROM read_parquet('{future_path}')
WHERE gsc_data_available IS TRUE
""").df()

print("Future rows:", len(future))

Future rows: 3901060


In [ ]:
for df in [march, future]:
    df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
    df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
    df["gsc_avg_position"] = df["gsc_avg_position"].fillna(999)
    df["sessions_ai"] = df["sessions_ai"].fillna(0)

    df["ctr"] = (
        df["gsc_clicks"] /
        df["gsc_impressions"].replace(0, np.nan)
    ).fillna(0)

In [ ]:
march_content = (
    march
    .groupby("content_hash_id", as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_ctr=("ctr", "mean"),
        march_position=("gsc_avg_position", "mean"),
        march_ai=("sessions_ai", "sum")
    )
)

future_content = (
    future
    .groupby("content_hash_id", as_index=False)
    .agg(
        future_impressions=("gsc_impressions", "sum"),
        future_clicks=("gsc_clicks", "sum"),
        future_ai=("sessions_ai", "sum")
    )
)

model_df = march_content.merge(
    future_content,
    on="content_hash_id",
    how="inner"
)

print("Matched content rows:", len(model_df))

Matched content rows: 158549


In [ ]:
model_df["future_click_change"] = (
    (
        model_df["future_clicks"] -
        model_df["march_clicks"]
    )
    /
    model_df["march_clicks"].replace(0, np.nan)
)

model_df["future_click_change"] = (
    model_df["future_click_change"]
    .replace([np.inf, -np.inf], np.nan)
)

display(
    model_df["future_click_change"]
    .describe(
        percentiles=[0.25, 0.50, 0.75]
    )
)

,future_click_change
count,68040.000000
mean,-0.086243
std,2.361239
min,-1.000000
25%,-1.000000
50%,-0.500000
75%,0.000000
max,289.500000


In [ ]:
change_cutoff = model_df["future_click_change"].quantile(0.25)

model_df["target"] = (
    model_df["future_click_change"] <= change_cutoff
).astype(int)

print("Outcome cutoff:", change_cutoff)
print("\nTarget distribution:")
print(model_df["target"].value_counts())
print("\nTarget proportion:")
print(model_df["target"].value_counts(normalize=True))

Outcome cutoff: -1.0

Target distribution:
target
0    138141
1     20408
Name: count, dtype: int64

Target proportion:
target
0    0.871283
1    0.128717
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

I will train the Random Forest using only the March signals used before the future outcome was observed.

The Week-4 baseline will be recreated using its original visibility, low-CTR, and AI-referral scoring rule.

Both approaches will be evaluated on the same content rows, using the same future outcome and the same precision, recall, and F1 metrics.

The comparison is intended to measure whether the learned model adds useful signal beyond the existing rule-based baseline.

In [ ]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_position",
    "march_ai"
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

print("Features:")
print(feature_cols)

print("\nRows:", len(X))
print("Positive outcomes:", int(y.sum()))

Features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_position', 'march_ai']

Rows: 158549
Positive outcomes: 20408


In [ ]:
# Keep the split deterministic.
# The future period defines the outcome, while March supplies the features.
#
# We additionally hold out 20% of the March-feature rows so the learned
# model is evaluated on observations it did not train on.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 126839
Test rows: 31710


In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (
    model_probability >= 0.50
).astype(int)

print("Random Forest trained.")

Random Forest trained.


In [ ]:
baseline_test = model_df.loc[X_test.index].copy()

baseline_test["visibility_score"] = (
    baseline_test["march_impressions"].rank(pct=True)
)

baseline_test["low_ctr_score"] = (
    1 - baseline_test["march_ctr"].rank(pct=True)
)

baseline_test["ai_score"] = (
    baseline_test["march_ai"].rank(pct=True)
)

baseline_test["baseline_score"] = (
    0.50 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["low_ctr_score"]
    + 0.20 * baseline_test["ai_score"]
)

In [ ]:
review_budget = max(
    1,
    int(len(baseline_test) * 0.10)
)

baseline_prediction = np.zeros(
    len(baseline_test),
    dtype=int
)

top_baseline_positions = (
    baseline_test["baseline_score"]
    .nlargest(review_budget)
    .index
)

baseline_prediction[
    baseline_test.index.get_indexer(top_baseline_positions)
] = 1

In [ ]:
baseline_precision = precision_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

model_precision = precision_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_f1 = f1_score(
    y_test,
    model_prediction,
    zero_division=0
)

comparison = pd.DataFrame({
    "approach": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "precision": [
        baseline_precision,
        model_precision
    ],
    "recall": [
        baseline_recall,
        model_recall
    ],
    "f1": [
        baseline_f1,
        model_f1
    ]
})

display(comparison.round(4))

,approach,precision,recall,f1
0,Week-4 baseline,0.0697,0.0541,0.0609
1,Random Forest,0.4659,0.9591,0.6272


## 4. Errors and interpretation

I will inspect false positives and false negatives rather than relying only on the overall metric.

A false positive means the model prioritized an observation that did not meet the measured future outcome. A false negative means the observation met the outcome but the model did not prioritize it.

I will also inspect permutation importance to understand which observed features the model relied on most.

Feature importance is directional. It does not show that changing a feature will cause the future outcome to change.

In [ ]:
error_df = model_df.loc[X_test.index].copy()

error_df["prediction"] = model_prediction
error_df["probability"] = model_probability

error_df["error_type"] = "correct"

error_df.loc[
    (error_df["prediction"] == 1) &
    (error_df["target"] == 0),
    "error_type"
] = "false_positive"

error_df.loc[
    (error_df["prediction"] == 0) &
    (error_df["target"] == 1),
    "error_type"
] = "false_negative"

print("Error counts:")
display(
    error_df["error_type"]
    .value_counts()
    .to_frame("count")
)

Error counts:


,count
error_type,
correct,27055
false_positive,4488
false_negative,167


In [ ]:
print("False positives:")
display(
    error_df[
        error_df["error_type"] == "false_positive"
    ][
        feature_cols +
        ["target", "probability"]
    ].head(10)
)

print("False negatives:")
display(
    error_df[
        error_df["error_type"] == "false_negative"
    ][
        feature_cols +
        ["target", "probability"]
    ].head(10)
)

False positives:


,march_impressions,march_clicks,march_ctr,march_position,march_ai,target,probability
27955,525,1,0.002155,8.576575,0,0,0.905972
156620,22,1,0.023810,4.940476,0,0,0.954255
141784,2523,2,0.000796,4.705274,0,0,0.719783
138161,493,2,0.003242,6.974753,0,0,0.850324
31177,122,1,0.009524,10.584736,0,0,0.945822
35673,352,4,0.008588,7.875306,0,0,0.725598
118691,276,3,0.007184,6.776587,0,0,0.818624
103341,3289,5,0.001794,23.453874,4,0,0.547106
97898,628,1,0.001613,6.500782,0,0,0.895005
2034,1405,3,0.002064,17.990889,0,0,0.718723


False negatives:


,march_impressions,march_clicks,march_ctr,march_position,march_ai,target,probability
70657,5480,12,0.003218,27.931954,0,1,0.106091
29780,964,6,0.006839,7.866635,0,1,0.425928
120124,4921,11,0.001531,9.523884,0,1,0.214051
154905,2747,6,0.002326,12.951237,0,1,0.457195
29355,3362,12,0.004271,6.894637,0,1,0.170376
82727,1015,6,0.006014,7.217369,0,1,0.394156
37374,2946,9,0.003156,3.014815,0,1,0.200998
117201,1766,7,0.005388,15.314273,0,1,0.415476
152187,4594,5,0.001112,33.701643,0,1,0.487141
91213,1208,7,0.004711,4.666147,0,1,0.386144


In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="f1"
)

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

display(importance)

,feature,importance_mean,importance_std
2,march_ctr,0.361646,0.002833
1,march_clicks,0.134120,0.004629
0,march_impressions,0.030016,0.000587
3,march_position,0.002210,0.000492
4,march_ai,0.000314,0.000069


## Observed result
The Random Forest achieved an F1 score of 0.6272 compared with 0.0609 for the Week-4 baseline on the same held-out observations. The model therefore showed a substantially higher measured F1 in this evaluation. False positives are observations the model prioritized that did not meet the measured future outcome, while false negatives are observations that met the outcome but were not prioritized. Permutation importance is interpreted as directional evidence about which March features were useful to the model, not as evidence of causation.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.